In [ ]:
#PROBLREM: PART OF MY RESEARCH IS TO MODEL AND OPTIMISE AN ADVANCED MANUFACTURING PROCESS CALLED CO2 LASER CUTTING.
#THIS PROCESS WAS USED TO CUT 29 SIMILAR GEAR USING USING 4 INPUTS:LASER POWER, CUTTING SPEED, FOCAL POSITION AND 
#GAS PRESSURE. THE OUTPUTS WERE SURFACE ROUGHNESS(RZ AND RA), MATERIAL REMOVAL RATE (MRR) AND DIMENSIONAL DEVIATIONS(DD)
#SVR WAS USED BECAUSE OUR DATA WAS NOT CORRELATING - IT WAS ALL OVER THE PLACE.
#THIS MODEL PREDICTS RA USING THE 4 INPUTS

In [1]:
import pandas as pd
import numpy as np

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
import seaborn as sns

In [4]:
df =pd.read_excel('Laser.xlsx')
df.head(29)

,Std,Run,Laser power [W],Cutting speed [m/min],Focal position [mm],Gas pressure [bar],Rz [µm],Ra [µm],MRR,DD [%]
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[mm3/min],NaN
1,1.0,11.0,1500.0,1.0,-2.5,13.0,3.69,0.74,2020.3,1.0264
2,2.0,24.0,2500.0,1.0,-2.5,13.0,2.21,0.43,2367.81,1.0268
3,3.0,28.0,1500.0,3.0,-2.5,13.0,2.36,0.47,5353.33,1.5244
4,4.0,6.0,2500.0,3.0,-2.5,13.0,3.40,0.76,6189.06,0.9088
5,5.0,14.0,2000.0,2.0,-3.5,10.0,4.74,0.93,4539.1,1.6582
6,6.0,9.0,2000.0,2.0,-1.5,10.0,4.33,0.89,3310.71,1.2305
7,7.0,26.0,2000.0,2.0,-3.5,16.0,4.58,0.96,4283.61,1.0342
8,8.0,5.0,2000.0,2.0,-1.5,16.0,3.42,0.67,4538.32,1.5174
9,9.0,17.0,1500.0,2.0,-2.5,10.0,3.15,0.59,3241.24,1.3131


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Std                    29 non-null     float64
 1   Run                    29 non-null     float64
 2   Laser power [W]        29 non-null     float64
 3   Cutting speed [m/min]  29 non-null     float64
 4   Focal position [mm]    29 non-null     float64
 5   Gas pressure [bar]     29 non-null     float64
 6   Rz [µm]                29 non-null     float64
 7   Ra [µm]                29 non-null     float64
 8   MRR                    30 non-null     object 
 9   DD [%]                 29 non-null     float64
dtypes: float64(9), object(1)
memory usage: 2.5+ KB


In [6]:
df.dropna(inplace=True)
df.head(30)

,Std,Run,Laser power [W],Cutting speed [m/min],Focal position [mm],Gas pressure [bar],Rz [µm],Ra [µm],MRR,DD [%]
1,1.0,11.0,1500.0,1.0,-2.5,13.0,3.69,0.74,2020.3,1.0264
2,2.0,24.0,2500.0,1.0,-2.5,13.0,2.21,0.43,2367.81,1.0268
3,3.0,28.0,1500.0,3.0,-2.5,13.0,2.36,0.47,5353.33,1.5244
4,4.0,6.0,2500.0,3.0,-2.5,13.0,3.40,0.76,6189.06,0.9088
5,5.0,14.0,2000.0,2.0,-3.5,10.0,4.74,0.93,4539.1,1.6582
6,6.0,9.0,2000.0,2.0,-1.5,10.0,4.33,0.89,3310.71,1.2305
7,7.0,26.0,2000.0,2.0,-3.5,16.0,4.58,0.96,4283.61,1.0342
8,8.0,5.0,2000.0,2.0,-1.5,16.0,3.42,0.67,4538.32,1.5174
9,9.0,17.0,1500.0,2.0,-2.5,10.0,3.15,0.59,3241.24,1.3131
10,10.0,1.0,2500.0,2.0,-2.5,10.0,3.17,0.65,4502.19,0.9945


In [7]:
df.describe()

,Std,Run,Laser power [W],Cutting speed [m/min],Focal position [mm],Gas pressure [bar],Rz [µm],Ra [µm],DD [%]
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,15.000000,15.000000,2000.000000,2.000000,-2.500000,13.000000,3.394828,0.688966,1.180128
std,8.514693,8.514693,327.326835,0.654654,0.654654,1.963961,0.828793,0.171034,0.262994
min,1.000000,1.000000,1500.000000,1.000000,-3.500000,10.000000,2.210000,0.430000,0.735100
25%,8.000000,8.000000,2000.000000,2.000000,-2.500000,13.000000,2.790000,0.570000,0.994500
50%,15.000000,15.000000,2000.000000,2.000000,-2.500000,13.000000,3.210000,0.660000,1.058200
75%,22.000000,22.000000,2000.000000,2.000000,-2.500000,13.000000,3.890000,0.820000,1.344500
max,29.000000,29.000000,2500.000000,3.000000,-1.500000,16.000000,4.980000,1.030000,1.914200


In [8]:
X = df[['Laser power [W]', 'Cutting speed [m/min]', 'Focal position [mm]', 'Gas pressure [bar]']]
y= df[ 'Ra [µm]']

In [9]:
sns.distplot(df['Ra [µm]'])

C:\Users\stukg\AppData\Local\Temp\ipykernel_23716\3959719262.py:1: UserWarning: 

`distplot` is a deprecated function and will be removed in seaborn v0.14.0.

Please adapt your code to use either `displot` (a figure-level function with
similar flexibility) or `histplot` (an axes-level function for histograms).

For a guide to updating your code to use the new functions, please see
https://gist.github.com/mwaskom/de44147ed2974457ad6372750bbe5751

  sns.distplot(df['Ra [µm]'])


<Axes: xlabel='Ra [µm]', ylabel='Density'>

# SCALING THE INPUTS

In [10]:
from sklearn.preprocessing import MinMaxScaler

In [11]:
scaler=MinMaxScaler()

In [12]:
X=scaler.fit_transform(X)

# SPLITTING THE DATA AND RUNNING THE MODEL

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=101)

In [15]:
from sklearn.svm import SVR

In [16]:
mdl = SVR()

In [17]:
mdl.fit(X_train, y_train)

SVR()

In [18]:
pred = mdl.predict(X_test)

# USING GRID SEARCH TO FIND OPTMAL HYPERPARAMETER

In [19]:
from sklearn.model_selection import GridSearchCV

In [20]:
param_grid = {'C':[0.01,0.1,1,10,100,1000],'gamma':[10,1,0.1,0.001,0.0001,0.00001],'epsilon':[0.001,0.001,0.01,0.1,1,10]}

In [21]:
grid = GridSearchCV(SVR(),param_grid,verbose =3)

In [22]:
grid.fit(X_train,y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
[CV 1/5] END ..C=0.01, epsilon=0.001, gamma=10;, score=-0.780 total time=   0.0s
[CV 2/5] END ..C=0.01, epsilon=0.001, gamma=10;, score=-0.873 total time=   0.0s
[CV 3/5] END ..C=0.01, epsilon=0.001, gamma=10;, score=-0.024 total time=   0.0s
[CV 4/5] END ..C=0.01, epsilon=0.001, gamma=10;, score=-0.064 total time=   0.0s
[CV 5/5] END ..C=0.01, epsilon=0.001, gamma=10;, score=-0.014 total time=   0.0s
[CV 1/5] END ...C=0.01, epsilon=0.001, gamma=1;, score=-0.693 total time=   0.0s
[CV 2/5] END ...C=0.01, epsilon=0.001, gamma=1;, score=-0.894 total time=   0.0s
[CV 3/5] END ....C=0.01, epsilon=0.001, gamma=1;, score=0.007 total time=   0.0s
[CV 4/5] END ...C=0.01, epsilon=0.001, gamma=1;, score=-0.147 total time=   0.0s
[CV 5/5] END ...C=0.01, epsilon=0.001, gamma=1;, score=-0.027 total time=   0.0s
[CV 1/5] END .C=0.01, epsilon=0.001, gamma=0.1;, score=-0.758 total time=   0.0s
[CV 2/5] END .C=0.01, epsilon=0.001, gamma=0.

GridSearchCV(estimator=SVR(),
             param_grid={'C': [0.01, 0.1, 1, 10, 100, 1000],
                         'epsilon': [0.001, 0.001, 0.01, 0.1, 1, 10],
                         'gamma': [10, 1, 0.1, 0.001, 0.0001, 1e-05]},
             verbose=3)

[CV 1/5] END ....C=10, epsilon=10, gamma=1e-05;, score=-0.859 total time=   0.0s
[CV 2/5] END ....C=10, epsilon=10, gamma=1e-05;, score=-0.732 total time=   0.0s
[CV 3/5] END ....C=10, epsilon=10, gamma=1e-05;, score=-0.024 total time=   0.0s
[CV 4/5] END ....C=10, epsilon=10, gamma=1e-05;, score=-0.069 total time=   0.0s
[CV 5/5] END ....C=10, epsilon=10, gamma=1e-05;, score=-0.014 total time=   0.0s
[CV 1/5] END ...C=100, epsilon=0.001, gamma=10;, score=-0.595 total time=   0.0s
[CV 2/5] END ...C=100, epsilon=0.001, gamma=10;, score=-0.761 total time=   0.0s
[CV 3/5] END ...C=100, epsilon=0.001, gamma=10;, score=-0.000 total time=   0.0s
[CV 4/5] END ...C=100, epsilon=0.001, gamma=10;, score=-0.040 total time=   0.0s
[CV 5/5] END ...C=100, epsilon=0.001, gamma=10;, score=-0.000 total time=   0.0s
[CV 1/5] END .....C=100, epsilon=0.001, gamma=1;, score=0.114 total time=   0.0s
[CV 2/5] END .....C=100, epsilon=0.001, gamma=1;, score=0.411 total time=   0.0s
[CV 3/5] END .....C=100, eps

[CV 5/5] END ...C=100, epsilon=10, gamma=1e-05;, score=-0.014 total time=   0.0s
[CV 1/5] END ..C=1000, epsilon=0.001, gamma=10;, score=-0.595 total time=   0.0s
[CV 2/5] END ..C=1000, epsilon=0.001, gamma=10;, score=-0.761 total time=   0.0s
[CV 3/5] END ..C=1000, epsilon=0.001, gamma=10;, score=-0.000 total time=   0.0s
[CV 4/5] END ..C=1000, epsilon=0.001, gamma=10;, score=-0.040 total time=   0.0s
[CV 5/5] END ..C=1000, epsilon=0.001, gamma=10;, score=-0.000 total time=   0.0s
[CV 1/5] END ....C=1000, epsilon=0.001, gamma=1;, score=0.114 total time=   0.0s
[CV 2/5] END ....C=1000, epsilon=0.001, gamma=1;, score=0.411 total time=   0.0s
[CV 3/5] END ....C=1000, epsilon=0.001, gamma=1;, score=0.412 total time=   0.0s
[CV 4/5] END ...C=1000, epsilon=0.001, gamma=1;, score=-0.112 total time=   0.0s
[CV 5/5] END ....C=1000, epsilon=0.001, gamma=1;, score=0.755 total time=   0.0s
[CV 1/5] END ..C=1000, epsilon=0.001, gamma=0.1;, score=0.631 total time=   0.0s
[CV 2/5] END ..C=1000, epsil

[CV 1/5] END C=1000, epsilon=0.1, gamma=0.0001;, score=-0.170 total time=   0.0s
[CV 2/5] END C=1000, epsilon=0.1, gamma=0.0001;, score=-0.421 total time=   0.0s
[CV 3/5] END C=1000, epsilon=0.1, gamma=0.0001;, score=-0.184 total time=   0.0s
[CV 4/5] END C=1000, epsilon=0.1, gamma=0.0001;, score=-0.318 total time=   0.0s
[CV 5/5] END .C=1000, epsilon=0.1, gamma=0.0001;, score=0.045 total time=   0.0s
[CV 1/5] END .C=1000, epsilon=0.1, gamma=1e-05;, score=-0.523 total time=   0.0s
[CV 2/5] END .C=1000, epsilon=0.1, gamma=1e-05;, score=-0.819 total time=   0.0s
[CV 3/5] END .C=1000, epsilon=0.1, gamma=1e-05;, score=-0.022 total time=   0.0s
[CV 4/5] END .C=1000, epsilon=0.1, gamma=1e-05;, score=-0.028 total time=   0.0s
[CV 5/5] END .C=1000, epsilon=0.1, gamma=1e-05;, score=-0.015 total time=   0.0s
[CV 1/5] END ......C=1000, epsilon=1, gamma=10;, score=-0.859 total time=   0.0s
[CV 2/5] END ......C=1000, epsilon=1, gamma=10;, score=-0.732 total time=   0.0s
[CV 3/5] END ......C=1000, e

GridSearchCV(estimator=SVR(),
             param_grid={'C': [0.01, 0.1, 1, 10, 100, 1000],
                         'epsilon': [0.001, 0.001, 0.01, 0.1, 1, 10],
                         'gamma': [10, 1, 0.1, 0.001, 0.0001, 1e-05]},
             verbose=3)

In [23]:
grid.best_params_

{'C': 100, 'epsilon': 0.01, 'gamma': 0.1}

In [24]:
grid_pred = grid.predict(X_test)

# MODEL EVALUATION

In [25]:
from sklearn import metrics

In [26]:
metrics.r2_score(y_test,grid_pred)

0.9271905823396333

In [27]:
metrics.mean_absolute_error(y_test,grid_pred)

0.02459594835971827

In [28]:
metrics.mean_squared_error(y_test,grid_pred)

0.0011881586844451105

In [29]:
np.sqrt(metrics.mean_squared_error(y_test,grid_pred))

0.03446967775371726

In [30]:
print(y_test)

20    0.45
25    0.56
8     0.67
28    0.61
3     0.47
27    0.59
23    0.89
26    0.58
Name: Ra [µm], dtype: float64


In [31]:
print(grid_pred)

[0.46338766 0.5895944  0.58890281 0.5895944  0.46420362 0.5895944
 0.85351365 0.5895944 ]


In [32]:
plt.scatter(y_test,grid_pred)
plt.xlabel('grid_pred')
plt.ylabel('y_test')
plt.title('y_test vs grid_predictions')

Text(0.5, 1.0, 'y_test vs grid_predictions')